# 📱 SMS Spam Detection — Text Mining Pipeline

---

**Dataset:** SMS Spam Collection (UCI ML Repository, ID 228)  
**Tugas:** Melatih satu model ML pada dataset publik bersih untuk mendemonstrasikan *(text) data mining*, lalu di-deploy ke Streamlit.

## Alur notebook
1. Data Understanding · 2. EDA · 3. Data Preparation (TF-IDF) · 4. Modeling · 5. Evaluation · 6. Interpretation · 7. Simpan model & visual

> `Runtime ▸ Run all`. Semua grafik otomatis tersimpan ke folder `assets/` untuk README.

## 0 · Setup

In [ ]:
!pip install -q ucimlrepo wordcloud joblib

import os, numpy as np, pandas as pd, joblib, json, warnings
import matplotlib.pyplot as plt, matplotlib as mpl, seaborn as sns
from wordcloud import WordCloud
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, f1_score, precision_recall_fscore_support)

# palet & folder output grafik
INK='#14181f'; SAFE='#1f9d6b'; SPAM='#e5484d'; ACC='#6c5ce7'; MUT='#6b7280'
mpl.rcParams.update({'figure.dpi':120,'axes.grid':True,'grid.color':'#eef0f2',
    'axes.axisbelow':True,'axes.titleweight':'bold'})
os.makedirs('assets', exist_ok=True)
RANDOM_STATE=42
print('Setup selesai ✅')

## 1 · Data Understanding

In [ ]:
def load_sms():
    try:
        from ucimlrepo import fetch_ucirepo
        ds=fetch_ucirepo(id=228)
        df=pd.DataFrame({'message':ds.data.features.iloc[:,0].astype(str),
                         'label':ds.data.targets.iloc[:,0].astype(str).str.lower()})
        print('Sumber: ucimlrepo ✅'); return df
    except Exception as e:
        print('ucimlrepo gagal (%s), pakai arsip UCI...'%type(e).__name__)
    import urllib.request, zipfile, io
    raw=urllib.request.urlopen('https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip',timeout=120).read()
    z=zipfile.ZipFile(io.BytesIO(raw))
    rows=[l.split('\t',1) for l in z.read('SMSSpamCollection').decode('utf-8').strip().split('\n')]
    return pd.DataFrame(rows,columns=['label','message'])

df=load_sms(); print('Dimensi:',df.shape); df.head()

In [ ]:
print('Distribusi label:'); print(df['label'].value_counts())
print('\nProporsi spam : %.1f%%'%(100*(df.label=='spam').mean()))
print('Missing value : %d'%df.isna().sum().sum())
print('Pesan duplikat: %d'%df.duplicated().sum())

## 2 · Exploratory Data Analysis

### 2a. Distribusi kelas (timpang → nilai pakai F1, bukan akurasi)

In [ ]:
cnt=df['label'].value_counts()
fig,ax=plt.subplots(figsize=(5.2,3.6))
ax.bar(cnt.index,cnt.values,color=[SAFE,SPAM],width=.6)
for i,v in enumerate(cnt.values): ax.text(i,v+40,str(v),ha='center',fontweight='bold')
ax.set_title('Distribusi Kelas: Ham vs Spam'); ax.set_ylabel('Jumlah pesan')
plt.tight_layout(); plt.savefig('assets/01_class_distribution.png',bbox_inches='tight'); plt.show()

### 2b. Panjang pesan

In [ ]:
df['len']=df['message'].str.len()
fig,ax=plt.subplots(figsize=(6.4,3.6))
for lbl,col in [('ham',SAFE),('spam',SPAM)]:
    d=df[df.label==lbl]['len']
    ax.hist(d,bins=40,range=(0,300),alpha=.55,color=col,density=True,label=f'{lbl} (rata2 {d.mean():.0f})')
ax.set_title('Distribusi Panjang Pesan'); ax.set_xlabel('karakter'); ax.set_ylabel('densitas'); ax.legend()
plt.tight_layout(); plt.savefig('assets/02_message_length.png',bbox_inches='tight'); plt.show()

### 2c. Word cloud

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
stop={'to','you','the','a','and','is','in','i','u','of','for','my','your','me','it','on'}
for a,lbl,cmap in [(ax[0],'ham','Greens'),(ax[1],'spam','Reds')]:
    wc=WordCloud(width=520,height=340,background_color='white',stopwords=stop,colormap=cmap).generate(' '.join(df[df.label==lbl]['message']))
    a.imshow(wc); a.axis('off'); a.set_title(lbl.upper(),fontsize=14)
plt.tight_layout(); plt.savefig('assets/03_wordclouds.png',bbox_inches='tight'); plt.show()

## 3 · Data Preparation — TF-IDF

Teks diubah menjadi vektor angka dengan **TF-IDF** (unigram+bigram, buang stopword Inggris, `min_df=2`). Split 80/20 *stratified*; vectorizer dipasang dalam `Pipeline` untuk mencegah *data leakage*.

In [ ]:
X=df['message'].astype(str); y=(df['label']=='spam').astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=RANDOM_STATE,stratify=y)
print('Train:',X_train.shape[0],'| Test:',X_test.shape[0])
tfidf=TfidfVectorizer(lowercase=True,stop_words='english',ngram_range=(1,2),min_df=2,sublinear_tf=True)

## 4 · Modeling — banding baseline lalu pilih satu

In [ ]:
def mk(clf): return Pipeline([('tfidf',tfidf),('clf',clf)])
cands={'Naive Bayes':MultinomialNB(alpha=0.1),
       'Logistic Regression':LogisticRegression(max_iter=1000,class_weight='balanced',C=10)}
scores={}
for n,cl in cands.items():
    cv=cross_val_score(mk(cl),X,y,cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE),scoring='f1',n_jobs=-1)
    scores[n]=(cv.mean(),cv.std()); print(f'{n:20s} CV spam-F1 = {cv.mean():.4f} ± {cv.std():.4f}')

fig,ax=plt.subplots(figsize=(5.6,3.2))
nm=list(scores); mv=[scores[n][0] for n in nm]; er=[scores[n][1] for n in nm]
ax.barh(nm,mv,xerr=er,color=[MUT,ACC],height=.5,capsize=5)
for i,m in enumerate(mv): ax.text(m-0.03,i,f'{m:.3f}',va='center',ha='right',color='white',fontweight='bold')
ax.set_xlim(0.8,0.98); ax.set_title('Perbandingan Model (CV Spam-F1)'); ax.set_xlabel('spam-F1')
plt.tight_layout(); plt.savefig('assets/04_model_comparison.png',bbox_inches='tight'); plt.show()

> **Logistic Regression** dipilih: F1 stabil, precision/recall seimbang, koefisien mudah diinterpretasi.

In [ ]:
model=mk(LogisticRegression(max_iter=1000,class_weight='balanced',C=10)).fit(X_train,y_train)
print('Model final terlatih ✅')

## 5 · Evaluation

In [ ]:
y_pred=model.predict(X_test); y_proba=model.predict_proba(X_test)[:,1]
print(classification_report(y_test,y_pred,target_names=['ham','spam'],digits=3))
print('ROC-AUC:',round(roc_auc_score(y_test,y_proba),4))

### 5a. Confusion matrix & kurva ROC

In [ ]:
fig,ax=plt.subplots(figsize=(4.8,4.2))
ConfusionMatrixDisplay(confusion_matrix(y_test,y_pred),display_labels=['ham','spam']).plot(ax=ax,cmap='Purples',colorbar=False,values_format='d')
ax.set_title('Confusion Matrix'); plt.tight_layout(); plt.savefig('assets/05_confusion_matrix.png',bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(5,4.2))
fpr,tpr,_=roc_curve(y_test,y_proba); auc=roc_auc_score(y_test,y_proba)
ax.plot(fpr,tpr,color=SPAM,lw=2.4,label=f'ROC (AUC={auc:.3f})'); ax.fill_between(fpr,tpr,alpha=.08,color=SPAM)
ax.plot([0,1],[0,1],'--',color=MUT); ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('Kurva ROC'); ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig('assets/06_roc_curve.png',bbox_inches='tight'); plt.show()

### 5b. Metrik per kelas & sebaran skor

In [ ]:
p,r,f,_=precision_recall_fscore_support(y_test,y_pred,labels=[0,1])
fig,ax=plt.subplots(figsize=(6,3.6)); x=np.arange(2); w=.25
ax.bar(x-w,p,w,label='Precision',color=ACC); ax.bar(x,r,w,label='Recall',color=SAFE); ax.bar(x+w,f,w,label='F1',color=SPAM)
ax.set_xticks(x); ax.set_xticklabels(['ham','spam']); ax.set_ylim(0,1.08); ax.legend(ncol=3,fontsize=9)
ax.set_title('Metrik per Kelas')
plt.tight_layout(); plt.savefig('assets/07_per_class_metrics.png',bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(6.4,3.6))
ax.hist(y_proba[y_test==0],bins=30,alpha=.6,color=SAFE,density=True,label='ham asli')
ax.hist(y_proba[y_test==1],bins=30,alpha=.6,color=SPAM,density=True,label='spam asli')
ax.axvline(0.5,color=INK,ls='--',label='ambang 0.5'); ax.set_title('Sebaran Skor Spam'); ax.set_xlabel('probabilitas spam'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('assets/09_score_distribution.png',bbox_inches='tight'); plt.show()

## 6 · Interpretation — kata pemicu

In [ ]:
vec=model.named_steps['tfidf']; clf=model.named_steps['clf']
names=np.array(vec.get_feature_names_out()); coef=clf.coef_[0]
ts=names[np.argsort(coef)[-15:]]; tsv=np.sort(coef)[-15:]
th=names[np.argsort(coef)[:15]]; thv=np.abs(np.sort(coef)[:15])
fig,ax=plt.subplots(1,2,figsize=(11,4.6))
ax[0].barh(ts,tsv,color=SPAM); ax[0].set_title('Kata Penanda SPAM'); ax[0].set_xlabel('bobot')
ax[1].barh(th,thv,color=SAFE); ax[1].set_title('Kata Penanda HAM'); ax[1].set_xlabel('bobot (abs)')
plt.tight_layout(); plt.savefig('assets/08_top_words.png',bbox_inches='tight'); plt.show()
print('Top spam:',list(names[np.argsort(coef)[-10:]][::-1]))

> **Insight:** spam bertumpu pada ajakan transaksi & nomor pendek (*txt, claim, won, prize, 150p, www*); ham berisi kosakata percakapan (*ok, home, hey, later*).

## 7 · Simpan model & unduh visual

In [ ]:
metrics={'spam_f1':float(f1_score(y_test,y_pred)),'accuracy':float((y_pred==y_test).mean()),'roc_auc':float(roc_auc_score(y_test,y_proba))}
joblib.dump({'model':model,'metrics':metrics},'sms_spam_model.joblib',compress=3)
print('Tersimpan: sms_spam_model.joblib',metrics)

# arsipkan folder assets biar mudah diunduh
import shutil; shutil.make_archive('assets','zip','assets'); print('assets.zip dibuat')

try:
    from google.colab import files
    files.download('sms_spam_model.joblib'); files.download('assets.zip')
except Exception: print('Bukan Colab — berkas ada di direktori kerja.')

---
## Ringkasan
| Tahap | Hasil |
|---|---|
| Dataset | 5.572 SMS · 13,4% spam · 0 missing |
| Representasi | TF-IDF (unigram+bigram) |
| Model | Logistic Regression (balanced) |
| Spam-F1 | ~0,92 |
| Akurasi | ~0,98 |
| ROC-AUC | ~0,99 |